<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/02_FlashAttention_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**FlashAttention From Scratch: Tiled Online Softmax Simulation**

An implementation of the foundational principles behind Tri Dao's FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness paper.

The Problem:
Standard attention materializes a massive intermediate attention matrix $S = \frac{QK^T}{\sqrt{d}} \in \mathbb{R}^{N \times N}$ in the GPU's slow High Bandwidth Memory (HBM). As sequence lengths ($N$) scale, this creates a quadratic $O(N^2)$ memory bottleneck that chokes hardware capacity.

The Fix:
FlashAttention bypasses this by loading small tiles (blocks) of $Q, K,$ and $V$ into the GPU's fast, local SRAM cache, computing local attention pieces, and dynamically updating global statistics using online softmax tracking. The global $N \times N$ score matrix is never written to HBM.

In [ ]:
import torch
import math

# Set random seed for exact reproducibility
torch.manual_seed(42)

# Small dimensions to clearly trace tensor tracking transformations
N, d = 8, 4

Q = torch.randn(N, d, device="cpu")
K = torch.randn(N, d, device="cpu")
V = torch.randn(N, d, device="cpu")

print(f"Inputs initialized successfully. Sequence Length (N): {N}, Head Dim (d): {d}")

**Baseline: Vanilla Attention Execution**

To prove our implementation's mathematical accuracy, we first establish a baseline using standard attention mechanics. This function will explicitly materialize the full $N \times N$ matrix in memory.

In [ ]:
def vanilla_attention(Q, K, V):
    scale = 1.0 / math.sqrt(Q.shape[-1])

    # Materializes the full N x N matrix in main memory
    scores = torch.matmul(Q, K.t()) * scale
    attention_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)

    return output

baseline_output = vanilla_attention(Q, K, V)
print("Baseline attention computed. Ready for tiled tracking comparison.")

**Tiled Online Softmax Matrix Free Implementation**

Instead of computing global softmax denominators across the entire row upfront, we track running row-wise statistics ($M$ for the max value to preserve numerical stability, and $L$ for the running denominator sum) and rescale previous block outputs dynamically as we encounter new blocks.

In [ ]:
def flash_attention_online_softmax(Q, K, V, B_r=2, B_c=2):
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # Allocate global tracking structures
    O = torch.zeros_like(Q)
    L = torch.zeros(N, 1, device=Q.device)       # Running sum of exponentials (denominator)
    M = torch.full((N, 1), -float('inf'), device=Q.device) # Running row max values

    # Outer loop tracking Column Blocks (SRAM cache partition simulation for K, V)
    for j in range(0, N, B_c):
        K_block = K[j:j+B_c]
        V_block = V[j:j+B_c]

        # Inner loop tracking Row Blocks (SRAM cache partition simulation for Q)
        for i in range(0, N, B_r):
            Q_block = Q[i:i+B_r]

            # Local matrix multiplication segment
            S_block = scale * torch.matmul(Q_block, K_block.t())

            # Isolate local block statistics
            m_block, _ = torch.max(S_block, dim=-1, keepdim=True)
            P_block = torch.exp(S_block - m_block)
            l_block = torch.sum(P_block, dim=-1, keepdim=True)

            # Fetch old states for the current row segment
            m_old = M[i:i+B_r]
            l_old = L[i:i+B_r]
            O_old = O[i:i+B_r]

            # Execute online softmax math corrections
            m_new = torch.maximum(m_old, m_block)
            alpha_old = torch.exp(m_old - m_new)
            alpha_new = torch.exp(m_block - m_new)

            l_new = (alpha_old * l_old) + (alpha_new * l_block)

            # Rescale the output accumulated values continuously
            O_new = (O_old * l_old * alpha_old + alpha_new * torch.matmul(P_block, V_block)) / l_new

            # Write updated states back to global parameters
            M[i:i+B_r] = m_new
            L[i:i+B_r] = l_new
            O[i:i+B_r] = O_new

    return O

# Execute custom block-level tracking
tiled_output = flash_attention_online_softmax(Q, K, V, B_r=2, B_c=2)

**Rigorous Numerical Verification Pass**

We test for structural and numerical identity using an absolute tolerance floor to verify that our independent block logic mirrors the standard execution graph flawlessly.

In [ ]:
print("Baseline Output Sample:\n", baseline_output[0])
print("\nTiled Output Sample:\n", tiled_output[0])

equivalence_status = torch.allclose(baseline_output, tiled_output, atol=1e-6)
print(f"\nExact Mathematical Equivalence Confirmed: {equivalence_status}")

**Complexity Scaling Analysis**

Standard attention forces an $O(N^2)$ memory footprint due to the generation of the massive intermediate $N \times N$ matrix. Our tiled implementation simulates an $O(N)$ memory tracking layout by computing and updating running statistics on bounded sub-blocks.

In [ ]:
import time

def profile_scaling_trends():
    # Scale sequence lengths to see where memory layouts separate
    sequence_lengths = [512, 1024, 2048]
    head_dimension = 64

    print("Evaluating memory-footprint complexities...")
    print("-" * 65)
    print(f"{'Seq Length (N)':<15} | {'Vanilla Elements':<20} | {'Tiled Tile Size':<15}")
    print("-" * 65)

    for seq_len in sequence_lengths:
        # Vanilla attention has to allocate this many elements in memory at once
        vanilla_matrix_elements = seq_len * seq_len

        # Tiled approach only holds small block slices in immediate tracking memory
        tiled_active_elements = 2 * 2 # Block size B_r * B_c

        print(f"{seq_len:<15} | {vanilla_matrix_elements:<20,} | {tiled_active_elements:<15}")

profile_scaling_trends()

In [ ]:
import time
import torch
import matplotlib.pyplot as plt

# 1. Setup device and scaling parameters
device = "cuda" if torch.cuda.is_available() else "cpu"
head_dim = 64
sequence_lengths = [128, 256, 512, 1024, 2048]

# Warm-up tracking lists
vanilla_times = []
tiled_times = []
vanilla_mems = []
tiled_mems = []

print(f"Starting hardware profiling on target device: {device.upper()}...")
print("-" * 70)

# 2. Benchmarking Loop
for N in sequence_lengths:
    Q_test = torch.randn(N, head_dim, device=device)
    K_test = torch.randn(N, head_dim, device=device)
    V_test = torch.randn(N, head_dim, device=device)

    # --- BENCHMARK 1: Vanilla Attention ---
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    _ = vanilla_attention(Q_test, K_test, V_test)
    t1 = time.perf_counter()

    vanilla_times.append(t1 - t0)
    if device == "cuda":
        vanilla_mems.append(torch.cuda.max_memory_allocated() / (1024 ** 2))
    else:
        vanilla_mems.append((N * N) * 4 / (1024 ** 2))

    # --- BENCHMARK 2: Tiled Online Softmax (FlashAttention) ---
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    _ = flash_attention_online_softmax(Q_test, K_test, V_test, B_r=64, B_c=64)
    t1 = time.perf_counter()

    tiled_times.append(t1 - t0)
    if device == "cuda":
        tiled_mems.append(torch.cuda.max_memory_allocated() / (1024 ** 2))
    else:
        tiled_mems.append((N * head_dim * 2) * 4 / (1024 ** 2))

# 3. Create Narrative-Driven Matplotlib Graphs
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("My biggest surprise while rebuilding FlashAttention", fontsize=16, fontweight="bold", y=0.98)

# Plot 1: Execution Latency (The Surprising Speed Bottleneck)
ax[0].plot(sequence_lengths, vanilla_times, label="Vanilla Attention (O(N²))", marker="o", color="tomato", linewidth=2.5)
ax[0].plot(sequence_lengths, tiled_times, label="Tiled Online Softmax (Python)", marker="s", color="teal", linewidth=2.5)
ax[0].set_title("The speedup didn't work... (Interpreter Overhead)", fontsize=12, fontweight="semibold", pad=12, color="#333333")
ax[0].set_xlabel("Sequence Length (N)", fontsize=10)
ax[0].set_ylabel("Time (seconds)", fontsize=10)
ax[0].legend(frameon=True, facecolor="whitesmoke")
ax[0].grid(True, linestyle="--", alpha=0.6)

# ---------------- LEFT GRAPH ANNOTATION ---------------- #

last_tiled_time = tiled_times[-1]

ax[0].annotate(
    "Python interpreter\nbecomes the bottleneck",
    xy=(2048, last_tiled_time),
    xytext=(1500, last_tiled_time * 0.88),
    fontsize=9,
    color="#444444",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        lw=1.2,
        color="#555555",
        shrinkA=4,
        shrinkB=4,
    ),
)
# Plot 2: Peak Memory Allocation (The Worked Optimization)
ax[1].plot(sequence_lengths, vanilla_mems, label="Vanilla Attention", marker="o", color="tomato", linewidth=2.5)
ax[1].plot(sequence_lengths, tiled_mems, label="Tiled Online Softmax", marker="s", color="teal", linewidth=2.5)
ax[1].set_title("But the memory optimization worked perfectly!", fontsize=12, fontweight="semibold", pad=12, color="#333333")
ax[1].set_xlabel("Sequence Length (N)", fontsize=10)
ax[1].set_ylabel("Memory (MB)", fontsize=10)
ax[1].legend(frameon=True, facecolor="whitesmoke")
ax[1].grid(True, linestyle="--", alpha=0.6)

# ---------------- RIGHT GRAPH ANNOTATION ---------------- #

last_tiled_mem = tiled_mems[-1]

ax[1].annotate(
    "Linear memory growth",
    xy=(2048, last_tiled_mem),
    xytext=(1450, last_tiled_mem + 1.3),
    fontsize=9,
    color="#444444",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        lw=1.2,
        color="#555555",
        shrinkA=4,
        shrinkB=4,
    ),
)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("flash_attention_story.png", dpi=300)
plt.show()